# 7. What didn't help: other algorithms, tuning, and corrections

**Life Expectancy Predictor · notebook 7 of 7**

The natural response to "the model misses some countries by five years" is to try a better model. This notebook shows that the project did, and what happened. Four algorithms were raced on the honest test; the random forest's settings were tuned; and (in notebook 5) its predictions were corrected afterwards. None of it moved the needle. That is a result, not a failure: it tells us the limit is the information in the 13 inputs, not the method, and it tells us where to look next.

## Step 1: Four algorithms on the same honest test

**What we are looking for.** Whether a different kind of model does better on unseen countries. The contenders:

- **Random forest** (the deployed model): 200 averaged decision trees.
- **Gradient boosting** and **histogram gradient boosting**: trees built one after another, each correcting the last. Often the strongest choice on tabular data.
- **Linear regression**: the simplest possible model, a straight-line formula. Included as a baseline to show what the tree models add.

All four were tested with the same five country-based folds and the same leak-free clipping.

In [1]:
# Setup: find the project folder and load the project's own code and settings.
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from config_loader import load_config

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
config = load_config(str(ROOT / "config" / "who_life_expectancy.yaml"), base_dir=str(ROOT))

def results(name):
    """Load one of the saved results files in model/."""
    return json.loads((ROOT / "model" / f"{name}.json").read_text(encoding="utf-8"))

print("Project folder:", ROOT.name)
from data_prep import get_unclipped_df, fold_winsorizer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error

df = get_unclipped_df(config)
X = df[config.predictor_columns].to_numpy(); y = df[config.target_column].to_numpy(); groups = df["Country"].to_numpy()
mp = config.model_params
contenders = {
    "Random forest (deployed)": RandomForestRegressor(n_estimators=mp["n_estimators"], random_state=mp["random_state"], n_jobs=-1),
    "Gradient boosting": GradientBoostingRegressor(random_state=mp["random_state"]),
    "Histogram gradient boosting": HistGradientBoostingRegressor(random_state=mp["random_state"]),
    "Linear regression": LinearRegression(),
}
folds = list(GroupKFold(n_splits=5).split(X, y, groups))
rows, preds = [], {}
for name, est in contenders.items():
    oof = np.zeros(len(y)); fold_r2 = []
    for tr, te in folds:
        m = Pipeline([("clip", fold_winsorizer(config)), ("model", est)]).fit(X[tr], y[tr])
        oof[te] = m.predict(X[te]); fold_r2.append(r2_score(y[te], oof[te]))
    preds[name] = oof
    rows.append({"Model": name, "R² (unseen countries)": round(np.mean(fold_r2), 3), "Average miss (years)": round(mean_absolute_error(y, oof), 2)})
pd.DataFrame(rows).set_index("Model")

Project folder: nb


,R² (unseen countries),Average miss (years)
Model,,
Random forest (deployed),0.901,2.14
Gradient boosting,0.896,2.17
Histogram gradient boosting,0.901,2.08
Linear regression,0.825,2.88


**How to read it.** One row per algorithm, scored the same way as notebook 3.

**What it means.** The three tree-based models are tied at about R² 0.90 and 2.1 years: the differences between them (a few hundredths of a year) are smaller than the differences between folds. Linear regression is clearly worse (R² 0.82, 2.9 years) because the relationships are not straight lines: notebook 4 showed the HIV effect is a cliff, not a slope. So a tree model is the right family, and which tree model does not matter.

**What's next and why.** If the models are tied overall, are they also tied on the hard countries, or does one of them crack Somalia?

## Step 2: Do they all fail on the same countries?

**What we are looking for.** Whether the hard countries from notebook 5 are hard for every algorithm. If they are, the problem is in the data, not the model.

In [2]:
per_country = pd.DataFrame({name: pd.Series(np.abs(p - y)).groupby(groups).mean() for name, p in preds.items()})
worst_rf = per_country["Random forest (deployed)"].sort_values(ascending=False).head(18).index
display(per_country.loc[worst_rf[:8]].round(1))

top18 = {name: set(per_country[name].sort_values(ascending=False).head(18).index) for name in preds}
always = set.intersection(*top18.values())
print(f"Countries in the worst 18 for EVERY one of the four models ({len(always)}): {', '.join(sorted(always))}")
print(f"Rank correlation of per-country error with the random forest's: " +
      ", ".join(f"{n.split(' (')[0]} {per_country[n].corr(per_country['Random forest (deployed)'], method='spearman'):.2f}" for n in preds if not n.startswith('Random')))

,Random forest (deployed),Gradient boosting,Histogram gradient boosting,Linear regression
Somalia,11.5,9.0,12.3,9.1
Sierra Leone,6.2,4.8,6.4,10.1
Angola,5.5,5.3,5.2,4.4
Côte d'Ivoire,5.4,4.9,4.7,6.4
Swaziland,5.4,5.2,5.0,7.4
Republic of Korea,4.5,4.6,3.8,5.9
Ethiopia,4.4,4.5,5.1,6.9
Zimbabwe,4.2,3.8,3.9,6.3


Countries in the worst 18 for EVERY one of the four models (6): Côte d'Ivoire, Ethiopia, Republic of Korea, Sierra Leone, Somalia, Swaziland
Rank correlation of per-country error with the random forest's: Gradient boosting 0.78, Histogram gradient boosting 0.88, Linear regression 0.63


**How to read it.** The table shows the random forest's eight hardest countries and how far each algorithm misses them. Below it: the countries that land in the worst 18 for all four models, and how closely each model's ranking of hard countries matches the random forest's (1.0 = identical ranking).

**What it means.** Every algorithm misses Somalia by 9-12 years. Six countries are in the worst 18 for all four models, including the simple linear one. The boosting models rank the hard countries almost exactly as the forest does (correlation 0.8-0.9). If a country were hard because of a quirk of random forests, another algorithm would get it right; nobody does. The hard countries are hard because the inputs do not contain what makes them different.

**What's next and why.** Perhaps the forest just needs better settings.

## Step 3: Tuning the random forest

**What we are looking for.** A random forest has settings ("hyperparameters"): how many trees, how deep each tree may grow, how many rows a leaf must hold. The project searched 36 combinations with the honest test, then checked the winner on held-out countries.

In [3]:
tune = results("tuning_results")
print("Settings searched:", tune["param_grid"])
print(f"Combinations tried: {tune['n_combinations']}   Time: {tune['search_time_seconds']:.0f} seconds")
print(f"\nBest settings found: {tune['best_params']}")
print(f"R² on held-out countries, best settings:    {tune['best_test_r2']:.4f}")
print(f"R² on held-out countries, default settings: {tune['default_test_r2']:.4f}")
print(f"Improvement from tuning:                    {tune['improvement_over_default']:+.4f}")

allr = pd.DataFrame([{**r["params"], "cv_r2": r["cv_r2_mean"]} for r in tune["all_results"]])
print(f"\nSpread of R² across all 36 combinations: {allr.cv_r2.min():.3f} to {allr.cv_r2.max():.3f}")

Settings searched: {'n_estimators': [100, 200, 300], 'max_depth': [None, 10, 15, 20], 'min_samples_leaf': [1, 2, 4]}
Combinations tried: 36   Time: 62 seconds

Best settings found: {'max_depth': 20, 'min_samples_leaf': 2, 'n_estimators': 300}
R² on held-out countries, best settings:    0.9039
R² on held-out countries, default settings: 0.9030
Improvement from tuning:                    +0.0009

Spread of R² across all 36 combinations: 0.882 to 0.883


**How to read it.** The best combination against the project's default, scored on countries neither had seen, and the spread across every combination tried.

**What it means.** Tuning improved R² by 0.001. All 36 combinations land within 0.002 of each other. The random forest is famously insensitive to its settings on data like this, and here that is exactly what happened. The project keeps the defaults: they are simpler to explain, and a 0.001 gain would not survive a change of random seed.

One more detail from the tuning run is worth knowing: when the same search was scored with the *random-row* test from notebook 3, it picked a different winner. The leaky test does not just overstate accuracy; it can also point you at the wrong settings.

**What's next and why.** Put the three "didn't help" results together.

## Step 4: What this adds up to

**What we are looking for.** The conclusion of the whole project, drawn from notebooks 3-7 together.

In [4]:
shr = results("shrinkage_check")
hgb = rows[2]
summary = pd.DataFrame([
    ["A different algorithm (best: histogram gradient boosting)", 0.901, hgb["R² (unseen countries)"]],
    ["Tuning the forest's settings (36 combinations)", round(tune["default_test_r2"], 3), round(tune["best_test_r2"], 3)],
    ["Correcting predictions afterwards: linear stretch", shr["A_raw"]["r2"], shr["B_linear"]["r2"]],
    ["Correcting predictions afterwards: flexible curve", shr["A_raw"]["r2"], shr["C_isotonic"]["r2"]],
], columns=["What was tried", "R² before", "R² after"]).set_index("What was tried")
summary["Change"] = (summary["R² after"] - summary["R² before"]).round(3)
summary

,R² before,R² after,Change
What was tried,,,
A different algorithm (best: histogram gradient boosting),0.9010,0.9010,0.000
Tuning the forest's settings (36 combinations),0.9030,0.9040,0.001
Correcting predictions afterwards: linear stretch,0.9047,0.9041,-0.001
Correcting predictions afterwards: flexible curve,0.9047,0.8990,-0.006


**How to read it.** Every attempt to improve on the deployed model, each scored against its own baseline on the honest test. (The baselines differ slightly because each check used its own fold layout; what matters is the change.)

**What it means.** Nothing changes R² by more than 0.001 in the right direction, and the flexible correction makes it worse by 0.006. Three independent lines of evidence (different algorithms, different settings, after-the-fact corrections) all say the same thing: **the method is not the bottleneck; the inputs are.** The 13 WHO indicators carry about 90% of what determines a country's life expectancy, and the remaining 10% lives in things they do not measure: conflict, epidemics, and how reliable each country's reporting is.

That makes the next step clear, and it is the one the project recommends: bring in outside data on those three things, and test it the same honest way. It would also be the right answer in an interview to "how would you improve this?": not a fancier model, but better information for the countries where the model struggles.

**What's next and why.** This is the last notebook. The Findings page collects the conclusions from all seven in one place, and the project documentation records every decision and why it was made.

## Summary

| What was tried | Result |
| --- | --- |
| Three other algorithms | Tree models tied at R² 0.90; linear regression clearly worse (0.82) |
| Do they fail on the same countries? | Yes: six countries are in the worst 18 for all four, Somalia missed by 9-12 years by all |
| Tuning the forest (36 settings) | +0.001 R²; all settings within 0.01 of each other |
| Correcting predictions afterwards (notebook 5) | No gain; the flexible version is worse |
| Conclusion | The limit is the inputs, not the method. Next: outside data on conflict, epidemics and reporting quality |